# Model Optimization and Performance Improvement with TensorFlow and Keras

## Practical Deep Learning Notebook using CIFAR-10


## Learning objectives

By the end of the notebook, you should be able to:

1. Explain how gradient descent updates model parameters.
2. Distinguish batch gradient descent, stochastic gradient descent, and mini-batch gradient descent.
3. Explain the ideas behind SGD, Momentum, Nesterov Accelerated Gradient, AdaGrad, AdaDelta, RMSProp, and Adam.
4. Compare optimizers on the same CIFAR-10 CNN under controlled conditions.
5. Explain why learning rate is an important hyperparameter.
6. Explain batch normalization and implement it with Keras.
7. Detect and discuss overfitting using training and validation curves.
8. Use dropout and early stopping as regularization techniques.
9. Explain vanishing and exploding gradients and inspect gradient norms.
10. Distinguish model parameters from hyperparameters.
11. Explain grid search, random search, Bayesian optimization, and KerasTuner.
12. Build and evaluate a final improved CIFAR-10 model.

## How to use this notebook

The notebook has two execution modes.

- **Quick mode** is the default. It uses a smaller CIFAR-10 subset and fewer epochs so the experiments are practical during a classroom session.
- **Full mode** uses the complete training split and more epochs. Use it when you want stronger final results and have sufficient CPU/GPU time.

Change the configuration in the next cell before running the experiments.

> Important: the optimizer comparison is intentionally kept simple. Batch normalization, dropout, and architecture changes are introduced later so that students can isolate the effect of the optimizer first.

In [ ]:
# Configuration
RUN_MODE = "quick"          # "quick" or "full"
SEED = 42

# Quick mode keeps the notebook practical for a classroom session.
QUICK_TRAIN_SAMPLES = 12000
QUICK_VAL_SAMPLES = 2000
QUICK_TEST_SAMPLES = 3000
QUICK_EPOCHS = 3

# Full mode settings
FULL_EPOCHS = 10

# Training configuration
BATCH_SIZE = 128
LEARNING_RATE = 0.001
AUTOTUNE = None  # assigned after TensorFlow is imported

# Optional expensive section. Set to True only when you want to run KerasTuner.
RUN_TUNER = False
TUNER_MAX_TRIALS = 5
TUNER_EPOCHS = 5

print(f"RUN_MODE = {RUN_MODE}")
print(f"RUN_TUNER = {RUN_TUNER}")

RUN_MODE = quick
RUN_TUNER = False


## 1. Environment setup

For a fresh environment, install TensorFlow and the optional KerasTuner package.

Do not run installation commands every time you open the notebook. Install them once in the environment and then restart the kernel if required.

For Windows users, TensorFlow GPU support depends on the TensorFlow version and installation route. Recent TensorFlow releases generally use WSL2 for NVIDIA GPU workflows, while native Windows GPU support is associated with older TensorFlow versions. The model code in this notebook is platform-neutral.

In [ ]:
# Optional installation cell. Run only if the packages are not already installed.
# %pip install tensorflow keras keras-tuner matplotlib pandas numpy

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings("ignore")

keras.utils.set_random_seed(SEED)
AUTOTUNE = tf.data.AUTOTUNE

print("TensorFlow version:", tf.__version__)
print("Keras version:", keras.__version__)
print("Available devices:")
for device in tf.config.list_physical_devices():
    print(" ", device)

## 2. The optimization problem

A neural network contains trainable parameters such as weights and biases. During training, the model tries to minimize a loss function.

For a parameter $w$, a basic gradient descent update is

$$
w_{t+1} = w_t - \eta
rac{\partial L}{\partial w_t}
$$

where:

- $w_t$ is the current parameter value.
- $L$ is the loss.
- $
rac{\partial L}{\partial w_t}$ is the gradient of the loss with respect to the parameter.
- $\eta$ is the learning rate.

The optimizer determines how the gradient information is converted into parameter updates.

This distinction is important:

- **Parameters** are learned by the model.
- **Hyperparameters** are chosen before or around training, such as learning rate, batch size, dropout rate, and number of layers.

### A small numerical example

Suppose $w=2$, the current gradient is $0.4$, and the learning rate is $0.1$.

$$
w_{new} = 2 - (0.1)(0.4) = 1.96
$$

The parameter moves in the opposite direction of the gradient because the gradient points toward increasing loss locally.

## 3. Load CIFAR-10

CIFAR-10 contains 60,000 color images of size $32	imes32$ across 10 classes. Keras provides a direct loader for the dataset.

The original training set contains 50,000 images and the test set contains 10,000 images. We create a separate validation set from the training data and keep the official test set untouched until final evaluation.

In [ ]:
class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

(x_train_raw, y_train_raw), (x_test_raw, y_test_raw) = keras.datasets.cifar10.load_data()

y_train_raw = y_train_raw.squeeze()
y_test_raw = y_test_raw.squeeze()

print("Training data:", x_train_raw.shape, y_train_raw.shape)
print("Test data:", x_test_raw.shape, y_test_raw.shape)
print("Pixel range:", x_train_raw.min(), "to", x_train_raw.max())

In [ ]:
# Visualize representative CIFAR-10 images.
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train_raw[i])
    ax.set_title(class_names[y_train_raw[i]])
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Preprocess and create train/validation/test sets

Pixel values are integers from 0 to 255. We scale them to approximately $[0,1]$ using

$$
x_{scaled} =
rac{x}{255}
$$

We also create a validation split from the original training set. The test set remains separate so that it represents unseen data for the final evaluation.

In [ ]:
rng = np.random.default_rng(SEED)
indices = rng.permutation(len(x_train_raw))

x_train_shuffled = x_train_raw[indices]
y_train_shuffled = y_train_raw[indices]

VAL_SIZE = 5000
x_train_full = x_train_shuffled[:-VAL_SIZE]
y_train_full = y_train_shuffled[:-VAL_SIZE]
x_val_full = x_train_shuffled[-VAL_SIZE:]
y_val_full = y_train_shuffled[-VAL_SIZE:]

if RUN_MODE == "quick":
    x_train = x_train_full[:QUICK_TRAIN_SAMPLES]
    y_train = y_train_full[:QUICK_TRAIN_SAMPLES]
    x_val = x_val_full[:QUICK_VAL_SAMPLES]
    y_val = y_val_full[:QUICK_VAL_SAMPLES]
    x_test = x_test_raw[:QUICK_TEST_SAMPLES]
    y_test = y_test_raw[:QUICK_TEST_SAMPLES]
    EPOCHS = QUICK_EPOCHS
else:
    x_train, y_train = x_train_full, y_train_full
    x_val, y_val = x_val_full, y_val_full
    x_test, y_test = x_test_raw, y_test_raw
    EPOCHS = FULL_EPOCHS

x_train = x_train.astype("float32") / 255.0
x_val = x_val.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

print("Train:", x_train.shape, y_train.shape)
print("Validation:", x_val.shape, y_val.shape)
print("Test:", x_test.shape, y_test.shape)
print("Epochs:", EPOCHS)

In [ ]:
# tf.data pipelines make the training input pipeline explicit.
train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train))
train_ds = train_ds.shuffle(len(x_train), seed=SEED, reshuffle_each_iteration=True).batch(BATCH_SIZE).prefetch(AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((x_val, y_val)).batch(BATCH_SIZE).prefetch(AUTOTUNE)
test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test)).batch(BATCH_SIZE).prefetch(AUTOTUNE)

print(train_ds)

## 5. Baseline CNN

We start with a small CNN. The architecture is deliberately modest because the purpose of this notebook is optimization and performance improvement rather than architectural search.

The baseline contains:

1. Convolution + ReLU
2. Max pooling
3. Convolution + ReLU
4. Max pooling
5. Flatten
6. Dense + ReLU
7. Dense + softmax

The output layer has 10 units because CIFAR-10 has 10 classes.

In [ ]:
def build_cnn(use_batch_norm=False, dropout_rate=0.0, dense_units=128):
    model = keras.Sequential(name="cifar10_cnn")
    model.add(layers.Input(shape=(32, 32, 3)))
    model.add(layers.Conv2D(32, (3, 3), padding="same", activation="relu"))
    if use_batch_norm:
        model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))

    model.add(layers.Conv2D(64, (3, 3), padding="same", activation="relu"))
    if use_batch_norm:
        model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))

    model.add(layers.Flatten())
    model.add(layers.Dense(dense_units, activation="relu"))
    if dropout_rate > 0:
        model.add(layers.Dropout(dropout_rate))
    model.add(layers.Dense(10, activation="softmax"))
    return model

baseline_model = build_cnn()
baseline_model.summary()

### Compile the model

For this multi-class classification problem, we use sparse categorical cross-entropy because the target labels are integer class IDs.

The compile step connects three ideas:

- **Loss:** measures prediction error.
- **Optimizer:** updates the trainable parameters.
- **Metric:** reports a human-readable performance measure.

In [ ]:
baseline_model.compile(
    optimizer=keras.optimizers.SGD(learning_rate=LEARNING_RATE),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

baseline_model.optimizer

## 6. Baseline training and overfitting

We train the same baseline architecture before introducing optimization techniques. Training and validation curves will help us answer:

- Is the model learning?
- Is validation performance improving with training?
- Does training accuracy continue to improve while validation accuracy stops improving?
- Does validation loss start increasing while training loss continues to decrease?

Those patterns are useful evidence when discussing overfitting.

In [ ]:
start = time.perf_counter()
baseline_history = baseline_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    verbose=1
)
baseline_time = time.perf_counter() - start

baseline_test_loss, baseline_test_acc = baseline_model.evaluate(test_ds, verbose=0)
print(f"Baseline test loss: {baseline_test_loss:.4f}")
print(f"Baseline test accuracy: {baseline_test_acc:.4f}")
print(f"Training time: {baseline_time:.1f} seconds")

In [ ]:
def plot_history(history, title="Training history"):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(history.history["accuracy"], label="Train")
    axes[0].plot(history.history["val_accuracy"], label="Validation")
    axes[0].set_title("Accuracy")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Accuracy")
    axes[0].legend()

    axes[1].plot(history.history["loss"], label="Train")
    axes[1].plot(history.history["val_loss"], label="Validation")
    axes[1].set_title("Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].legend()

    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

plot_history(baseline_history, "Baseline CNN")

## 7. Gradient Descent, SGD, and mini-batches

There are three common ways to form the gradient used for an update:

| Method | Gradient calculated from | Main idea |
|---|---|---|
| Batch Gradient Descent | Entire training set | Stable but potentially expensive update |
| Stochastic Gradient Descent | One example | Frequent, noisy updates |
| Mini-batch Gradient Descent | A small batch | Practical compromise used by modern deep learning |

In Keras, `batch_size` controls the mini-batch size. Therefore, even when we say **SGD** in a neural-network experiment, the practical implementation normally uses mini-batch updates.

For a mini-batch $B$, the average gradient can be written as

$$

abla L_B(w) =
rac{1}{|B|}\sum_{i\in B}
abla L_i(w)
$$

## 8. Learning rate

The learning rate $\eta$ controls the size of each parameter update.

$$
w_{t+1} = w_t - \eta g_t
$$

A very small learning rate can make training slow. A very large learning rate can cause unstable updates or prevent useful convergence.

The correct learning rate depends on the model, data, optimizer, and other hyperparameters. We therefore treat it as a hyperparameter rather than a fixed universal constant.

In [ ]:
# A simple learning-rate comparison on the same architecture.
# Keep the experiment small in class; increase EPOCHS if you want a more reliable comparison.
learning_rates = [0.0001, 0.001, 0.01]
lr_results = []

for lr in learning_rates:
    keras.backend.clear_session()
    keras.utils.set_random_seed(SEED)
    model = build_cnn()
    model.compile(
        optimizer=keras.optimizers.SGD(learning_rate=lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        verbose=0
    )
    lr_results.append({
        "learning_rate": lr,
        "best_val_accuracy": max(history.history["val_accuracy"]),
        "final_val_loss": history.history["val_loss"][-1]
    })

lr_results_df = pd.DataFrame(lr_results)
lr_results_df

## 9. Optimizer family: the progression

The optimizers in this lesson can be understood as a progression of ideas:

1. **SGD:** use the current gradient.
2. **Momentum:** accumulate useful direction from previous updates.
3. **NAG:** use a look-ahead position before evaluating the gradient.
4. **AdaGrad:** adapt the learning rate separately for each parameter using historical squared gradients.
5. **AdaDelta:** reduce AdaGrad's dependence on an ever-growing history by using a moving window of recent squared gradients.
6. **RMSProp:** use a moving average of squared gradients to normalize update sizes.
7. **Adam:** combine momentum-like first-moment tracking with RMSProp-like second-moment tracking.

The exact equations differ by optimizer, but the teaching goal is the same: use gradient information more effectively than a plain fixed-step update.

## 10. SGD

The basic mini-batch SGD update is

$$
w_{t+1} = w_t - \eta g_t
$$

where $g_t$ is the gradient computed from the current mini-batch.

**Strength:** simple and easy to reason about.

**Limitation:** the path toward a minimum can be noisy, especially in directions where gradients change sharply.

## 11. SGD with Momentum

Momentum maintains a velocity-like quantity. A common form is

$$
v_t = eta v_{t-1} + g_t
$$

$$
w_{t+1} = w_t - \eta v_t
$$

The previous update direction contributes to the current update. This can smooth noisy movement and help the optimizer continue moving in useful directions.

## 12. Nesterov Accelerated Gradient (NAG)

NAG is a look-ahead version of momentum. Instead of computing the gradient only at the current parameter position, it considers the direction suggested by the momentum step before evaluating the correction.

Conceptually:

1. Look ahead using the current momentum.
2. Evaluate the gradient at that look-ahead point.
3. Use the gradient to update the momentum and parameters.

In Keras, NAG can be enabled through SGD with momentum:

```python
keras.optimizers.SGD(learning_rate=..., momentum=0.9, nesterov=True)
```

## 13. AdaGrad

AdaGrad gives each parameter an adaptive learning rate. It accumulates historical squared gradients.

A simplified form is

$$
G_t = G_{t-1} + g_t^2
$$

$$
w_{t+1} = w_t -
rac{\eta}{\sqrt{G_t}+\epsilon}g_t
$$

Parameters that repeatedly receive large gradients get progressively smaller effective steps.

A practical limitation is that the accumulated quantity can keep growing, eventually making the effective learning rate very small.

## 14. AdaDelta

AdaDelta was designed to address the continuously shrinking learning rates associated with AdaGrad. Instead of storing an ever-growing sum of squared gradients, it maintains an exponentially decaying average of recent squared gradients.

The important teaching idea is:

> AdaGrad uses accumulated history; AdaDelta focuses on a moving window of recent history.

## 15. RMSProp

RMSProp also maintains a moving average of squared gradients.

$$
E[g^2]_t =
ho E[g^2]_{t-1} + (1-
ho)g_t^2
$$

The update scales the gradient using this moving average:

$$
w_{t+1} = w_t -
rac{\eta}{\sqrt{E[g^2]_t}+\epsilon}g_t
$$

The effect is to reduce the step size for parameters experiencing large gradients and relatively increase it for parameters with smaller gradients.

## 16. Adam

Adam combines two ideas:

- a first-moment estimate, similar to momentum;
- a second-moment estimate, similar to RMSProp.

The simplified moment equations are

$$
m_t = eta_1m_{t-1} + (1-eta_1)g_t
$$

$$
v_t = eta_2v_{t-1} + (1-eta_2)g_t^2
$$

After bias correction, Adam uses these estimates to form an adaptive update.

Adam is widely used because it is a strong general-purpose baseline. The experiment below still treats its performance as an empirical result rather than assuming that it must always win.

## 17. Compare the optimizers fairly

This is the central experiment.

To compare optimizers fairly, keep the following fixed:

- CNN architecture
- initial weights
- training and validation data
- batch size
- number of epochs
- loss function
- random seed
- evaluation metric

Only the optimizer changes.

### Optimizers compared

1. SGD
2. SGD + Momentum
3. SGD + NAG
4. AdaGrad
5. AdaDelta
6. RMSProp
7. Adam

> Course note: the lesson material includes an MNIST comparison in which Adam is shown as performing efficiently relative to other optimizers. Here we do not transfer that result to CIFAR-10; we run a controlled CIFAR-10 experiment so students can observe the result for this setup.


In [ ]:
def make_optimizer(name, learning_rate=LEARNING_RATE):
    if name == "SGD":
        return keras.optimizers.SGD(learning_rate=learning_rate)
    if name == "Momentum":
        return keras.optimizers.SGD(learning_rate=learning_rate, momentum=0.9)
    if name == "NAG":
        return keras.optimizers.SGD(learning_rate=learning_rate, momentum=0.9, nesterov=True)
    if name == "AdaGrad":
        return keras.optimizers.Adagrad(learning_rate=learning_rate)
    if name == "AdaDelta":
        return keras.optimizers.Adadelta(learning_rate=learning_rate, rho=0.95)
    if name == "RMSProp":
        return keras.optimizers.RMSprop(learning_rate=learning_rate, rho=0.9)
    if name == "Adam":
        return keras.optimizers.Adam(learning_rate=learning_rate)
    raise ValueError(f"Unknown optimizer: {name}")

optimizer_names = ["SGD", "Momentum", "NAG", "AdaGrad", "AdaDelta", "RMSProp", "Adam"]

# Capture one common initialization so optimizer differences are not mixed with initialization differences.
keras.backend.clear_session()
keras.utils.set_random_seed(SEED)
reference_model = build_cnn()
initial_weights = reference_model.get_weights()

optimizer_histories = {}
optimizer_models = {}
optimizer_rows = []

for name in optimizer_names:
    keras.backend.clear_session()
    model = build_cnn()
    model.set_weights(initial_weights)
    model.compile(
        optimizer=make_optimizer(name),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    start = time.perf_counter()
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        verbose=0
    )
    elapsed = time.perf_counter() - start

    test_loss, test_acc = model.evaluate(test_ds, verbose=0)
    best_epoch = int(np.argmax(history.history["val_accuracy"]) + 1)

    optimizer_histories[name] = history.history
    optimizer_models[name] = model
    optimizer_rows.append({
        "Optimizer": name,
        "Final Train Accuracy": history.history["accuracy"][-1],
        "Best Validation Accuracy": max(history.history["val_accuracy"]),
        "Final Validation Loss": history.history["val_loss"][-1],
        "Best Validation Epoch": best_epoch,
        "Test Accuracy": test_acc,
        "Training Time (s)": elapsed
    })

optimizer_results = pd.DataFrame(optimizer_rows).sort_values("Best Validation Accuracy", ascending=False).reset_index(drop=True)
optimizer_results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, history in optimizer_histories.items():
    axes[0].plot(history["val_accuracy"], label=name)
    axes[1].plot(history["val_loss"], label=name)

axes[0].set_title("Validation Accuracy by Optimizer")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Validation Accuracy")
axes[0].legend()

axes[1].set_title("Validation Loss by Optimizer")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Validation Loss")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_df = optimizer_results.sort_values("Best Validation Accuracy", ascending=True)
axes[0].barh(plot_df["Optimizer"], plot_df["Best Validation Accuracy"])
axes[0].set_title("Best Validation Accuracy")
axes[0].set_xlabel("Accuracy")

axes[1].barh(plot_df["Optimizer"], plot_df["Training Time (s)"])
axes[1].set_title("Training Time")
axes[1].set_xlabel("Seconds")

plt.tight_layout()
plt.show()

### How to interpret the optimizer comparison

Do not ask only, "Which optimizer has the highest accuracy?"

Also ask:

- How quickly did validation accuracy improve?
- Was validation loss stable?
- Did training accuracy improve without a similar validation improvement?
- How much training time was required?
- Does the result remain consistent when the experiment is repeated?

The table is an empirical result for this model, dataset split, seed, and training configuration. It is not a universal ranking of optimizers.

## 18. Batch Normalization

Batch normalization normalizes layer activations using statistics computed from a mini-batch during training. A simplified normalization is

$$
\hat{x} =
rac{x-\mu_B}{\sqrt{\sigma_B^2+\epsilon}}
$$

The layer can then learn a scale and shift:

$$
y = \gamma\hat{x} + eta
$$

where $\gamma$ and $eta$ are trainable parameters.

In Keras, this is implemented with `layers.BatchNormalization()`.

The goal in this notebook is not to claim that batch normalization always improves every model. Instead, we will compare the baseline architecture with and without it.

In [ ]:
keras.backend.clear_session()
keras.utils.set_random_seed(SEED)

bn_model = build_cnn(use_batch_norm=True)
bn_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

start = time.perf_counter()
bn_history = bn_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    verbose=0
)
bn_time = time.perf_counter() - start

bn_test_loss, bn_test_acc = bn_model.evaluate(test_ds, verbose=0)

batch_norm_comparison = pd.DataFrame([
    {
        "Model": "Baseline CNN",
        "Best Validation Accuracy": max(baseline_history.history["val_accuracy"]),
        "Test Accuracy": baseline_test_acc,
        "Training Time (s)": baseline_time
    },
    {
        "Model": "CNN + Batch Normalization",
        "Best Validation Accuracy": max(bn_history.history["val_accuracy"]),
        "Test Accuracy": bn_test_acc,
        "Training Time (s)": bn_time
    }
])

batch_norm_comparison

## 19. Dropout

Dropout is a regularization technique. During training, a fraction of activations is randomly set to zero.

If the dropout rate is $p$, approximately a fraction $p$ of the selected activations is dropped during a training step. During inference, dropout is disabled and the layer uses the learned model without randomly removing activations.

Dropout is therefore different from an optimizer:

- the optimizer controls **how parameters are updated**;
- dropout changes the **training behavior of the network** to reduce reliance on particular activations.

In [ ]:
dropout_rates = [0.0, 0.2, 0.5]
dropout_rows = []
dropout_histories = {}

for rate in dropout_rates:
    keras.backend.clear_session()
    keras.utils.set_random_seed(SEED)
    model = build_cnn(dropout_rate=rate)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        verbose=0
    )
    test_loss, test_acc = model.evaluate(test_ds, verbose=0)

    dropout_histories[rate] = history.history
    dropout_rows.append({
        "Dropout Rate": rate,
        "Final Train Accuracy": history.history["accuracy"][-1],
        "Best Validation Accuracy": max(history.history["val_accuracy"]),
        "Test Accuracy": test_acc
    })

dropout_results = pd.DataFrame(dropout_rows)
dropout_results

In [ ]:
plt.figure(figsize=(10, 5))
for rate, history in dropout_histories.items():
    plt.plot(history["val_accuracy"], label=f"Dropout {rate}")
plt.title("Validation Accuracy with Different Dropout Rates")
plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.legend()
plt.show()

## 20. Early Stopping

Early stopping monitors a validation metric and stops training when improvement has stalled.

For example:

```python
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)
```

`patience=3` means training can continue for three consecutive epochs without improvement before stopping. `restore_best_weights=True` returns the model to the weights from the best monitored epoch.

Early stopping is useful when more epochs no longer improve validation performance. It also prevents unnecessary computation.

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

keras.backend.clear_session()
keras.utils.set_random_seed(SEED)

es_model = build_cnn(use_batch_norm=True, dropout_rate=0.2)
es_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

start = time.perf_counter()
es_history = es_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=max(EPOCHS, 8),
    callbacks=[early_stopping],
    verbose=0
)
es_time = time.perf_counter() - start

es_test_loss, es_test_acc = es_model.evaluate(test_ds, verbose=0)
print("Epochs actually trained:", len(es_history.history["loss"]))
print("Best validation accuracy:", max(es_history.history["val_accuracy"]))
print("Test accuracy:", es_test_acc)

## 21. Vanishing and exploding gradients

Deep networks are trained using repeated applications of the chain rule. If gradients become extremely small as they move backward through many layers, learning can become very slow. This is the **vanishing gradient** problem.

If gradients become excessively large, updates can become unstable. This is the **exploding gradient** problem.

Common strategies used in deep learning include suitable initialization, activation functions, normalization, architecture choices, and optimization techniques.

We can inspect gradient magnitudes directly for a mini-batch using `tf.GradientTape`.

In [ ]:
def gradient_norms(model, dataset):
    x_batch, y_batch = next(iter(dataset))
    with tf.GradientTape() as tape:
        predictions = model(x_batch, training=True)
        loss = keras.losses.sparse_categorical_crossentropy(y_batch, predictions)
        loss = tf.reduce_mean(loss)

    gradients = tape.gradient(loss, model.trainable_variables)
    rows = []
    for variable, gradient in zip(model.trainable_variables, gradients):
        if gradient is not None:
            rows.append({
                "Layer": variable.name,
                "Gradient L2 Norm": float(tf.linalg.global_norm([gradient]).numpy())
            })
    return pd.DataFrame(rows)

gradient_norm_table = gradient_norms(es_model, train_ds)
gradient_norm_table

### Reading gradient norms

There is no single numerical threshold that universally defines a vanishing or exploding gradient. The useful classroom question is whether gradient magnitudes are becoming extremely small or excessively large relative to the other layers and whether that behavior is affecting learning.

## 22. Parameters vs hyperparameters

### Parameters

Learned automatically during training:

- weights
- biases

### Hyperparameters

Chosen outside the normal parameter-update process:

- learning rate
- optimizer type
- batch size
- number of epochs
- number of layers
- number of neurons or filters
- dropout rate
- momentum coefficient
- regularization strength

Hyperparameter tuning is the process of searching for useful values of these choices using validation performance as feedback.

## 23. Manual tuning

A sensible tuning workflow starts with a small number of meaningful changes.

For example:

1. Establish a baseline.
2. Change the optimizer or learning rate.
3. Observe validation behavior.
4. Introduce batch normalization or dropout when appropriate.
5. Adjust architecture only when there is evidence that the architecture is limiting performance.
6. Keep the test set untouched while selecting hyperparameters.

Manual tuning is useful for learning because it makes cause and effect visible, but it becomes expensive when many hyperparameters interact.

## 24. Grid Search

Grid search defines a finite set of values for each hyperparameter and evaluates combinations systematically.

If there are $k$ hyperparameters and hyperparameter $i$ has $n_i$ candidate values, the number of combinations is

$$
N = \prod_{i=1}^{k} n_i
$$

For example, 3 learning rates and 2 dropout rates already produce

$$
3	imes2 = 6
$$

configurations.

Grid search is thorough over the specified grid, but the number of experiments can grow rapidly.

In [ ]:
# A small illustrative grid. This cell only creates the combinations; it does not train six models.
from itertools import product

grid = {
    "learning_rate": [0.0005, 0.001, 0.005],
    "dropout_rate": [0.2, 0.5]
}

grid_combinations = [
    dict(zip(grid.keys(), values))
    for values in product(*grid.values())
]

pd.DataFrame(grid_combinations)

## 25. Random Search

Random search samples hyperparameter combinations instead of evaluating every point in a fixed grid.

This can be computationally useful when only a few hyperparameters have a strong influence on the result. It also explores continuous ranges naturally when values are sampled from distributions.

The lesson material highlights random search as an automatic tuning approach alongside grid search and Bayesian optimization.

In [ ]:
# Illustrative random sampling. No model training is performed in this cell.
rng = np.random.default_rng(SEED)

random_candidates = pd.DataFrame({
    "learning_rate": 10 ** rng.uniform(-4, -2, size=6),
    "dropout_rate": rng.uniform(0.1, 0.6, size=6),
    "dense_units": rng.choice([64, 128, 256], size=6)
})

random_candidates

## 26. Bayesian Optimization

Bayesian optimization builds a probabilistic model of the relationship between hyperparameter configurations and validation performance. The search then uses previous results to decide which configuration is promising to evaluate next.

The practical intuition is:

> Do not search blindly. Use information from earlier experiments to choose the next experiment.

This can reduce the number of expensive model-training trials when the search space is large.

## 27. KerasTuner

KerasTuner provides a Keras-friendly interface for automated hyperparameter search. The lesson material uses KerasTuner with random search in its end project; here we adapt the practical dataset to CIFAR-10.

The tuner section is optional because automated tuning can be computationally expensive. Set `RUN_TUNER = True` in the configuration cell when you are ready.

The lesson also lists gradient-based tuning and evolutionary optimization among automatic approaches. They are introduced here conceptually; the executable automated search uses KerasTuner random search, with Bayesian optimization shown as an optional extension.


In [ ]:
if RUN_TUNER:
    try:
        import keras_tuner as kt
        print("KerasTuner version:", kt.__version__)
    except ImportError as exc:
        raise ImportError(
            "KerasTuner is not installed. Run: %pip install keras-tuner and restart the kernel."
        ) from exc
else:
    print("KerasTuner section is disabled. Set RUN_TUNER = True to run it.")

In [ ]:
if RUN_TUNER:
    def build_tuned_model(hp):
        filters_1 = hp.Choice("filters_1", values=[32, 48])
        filters_2 = hp.Choice("filters_2", values=[64, 96])
        dense_units = hp.Choice("dense_units", values=[64, 128, 256])
        dropout_rate = hp.Float("dropout_rate", min_value=0.1, max_value=0.5, step=0.1)
        learning_rate = hp.Float("learning_rate", min_value=1e-4, max_value=1e-2, sampling="log")

        model = keras.Sequential([
            layers.Input(shape=(32, 32, 3)),
            layers.Conv2D(filters_1, 3, padding="same", activation="relu"),
            layers.BatchNormalization(),
            layers.MaxPooling2D(),
            layers.Conv2D(filters_2, 3, padding="same", activation="relu"),
            layers.BatchNormalization(),
            layers.MaxPooling2D(),
            layers.Flatten(),
            layers.Dense(dense_units, activation="relu"),
            layers.Dropout(dropout_rate),
            layers.Dense(10, activation="softmax")
        ])

        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"]
        )
        return model

    tuner = kt.RandomSearch(
        build_tuned_model,
        objective="val_accuracy",
        max_trials=TUNER_MAX_TRIALS,
        overwrite=True,
        directory="keras_tuner_results",
        project_name="cifar10_random_search"
    )

    tuner.search(
        x_train,
        y_train,
        validation_data=(x_val, y_val),
        epochs=TUNER_EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=1
    )

    tuner_results = tuner.get_best_hyperparameters(num_trials=5)
    print("Best hyperparameter configuration:")
    for hp in tuner_results[:1]:
        for key, value in hp.values.items():
            print(f"  {key}: {value}")

## 28. Optional Bayesian search with KerasTuner

KerasTuner also provides a Bayesian optimization tuner. The API is similar to random search; the search strategy changes.

This is intentionally kept as an optional extension because running multiple tuning strategies can be expensive in a classroom environment.

In [ ]:
if RUN_TUNER:
    bayesian_tuner = kt.BayesianOptimization(
        build_tuned_model,
        objective="val_accuracy",
        max_trials=3,
        overwrite=True,
        directory="keras_tuner_results",
        project_name="cifar10_bayesian_search"
    )

    # Uncomment to run the Bayesian search.
    # bayesian_tuner.search(
    #     x_train,
    #     y_train,
    #     validation_data=(x_val, y_val),
    #     epochs=TUNER_EPOCHS,
    #     batch_size=BATCH_SIZE,
    #     verbose=1
    # )

    print("BayesianOptimization tuner created. Search call is commented to avoid an accidental second expensive search.")

## 29. Build the final improved model

The final model combines techniques that address different parts of the training problem:

- **Adam:** optimization strategy.
- **Batch normalization:** helps stabilize layer activations.
- **Dropout:** regularization to reduce over-reliance on particular activations.
- **Early stopping:** stops training when validation loss stops improving.

These techniques solve different problems, so they should not be described as interchangeable.

In [ ]:
# Use the best tuned hyperparameters when tuning was run; otherwise use a sensible classroom configuration.
if RUN_TUNER:
    best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]
    final_filters_1 = best_hp.get("filters_1")
    final_filters_2 = best_hp.get("filters_2")
    final_dense_units = best_hp.get("dense_units")
    final_dropout_rate = best_hp.get("dropout_rate")
    final_learning_rate = best_hp.get("learning_rate")
else:
    final_filters_1 = 32
    final_filters_2 = 64
    final_dense_units = 128
    final_dropout_rate = 0.2
    final_learning_rate = LEARNING_RATE

final_model = keras.Sequential([
    layers.Input(shape=(32, 32, 3)),
    layers.Conv2D(final_filters_1, 3, padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),
    layers.Conv2D(final_filters_2, 3, padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(final_dense_units, activation="relu"),
    layers.Dropout(final_dropout_rate),
    layers.Dense(10, activation="softmax")
])

final_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=final_learning_rate),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

final_model.summary()

In [ ]:
final_early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

final_epochs = max(EPOCHS, 8)

start = time.perf_counter()
final_history = final_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=final_epochs,
    callbacks=[final_early_stopping],
    verbose=1
)
final_time = time.perf_counter() - start

final_test_loss, final_test_acc = final_model.evaluate(test_ds, verbose=0)

print(f"Final test loss: {final_test_loss:.4f}")
print(f"Final test accuracy: {final_test_acc:.4f}")
print(f"Epochs actually trained: {len(final_history.history['loss'])}")
print(f"Training time: {final_time:.1f} seconds")

In [ ]:
plot_history(final_history, "Final Improved CIFAR-10 Model")

## 30. Before vs after

The comparison below brings the notebook together.

The baseline and final model are different in more than one way, so this is a **performance-improvement summary**, not a controlled optimizer comparison. The controlled optimizer experiment was completed earlier.

In [ ]:
before_after = pd.DataFrame([
    {
        "Metric": "Best Validation Accuracy",
        "Baseline": max(baseline_history.history["val_accuracy"]),
        "Final Improved": max(final_history.history["val_accuracy"])
    },
    {
        "Metric": "Test Accuracy",
        "Baseline": baseline_test_acc,
        "Final Improved": final_test_acc
    },
    {
        "Metric": "Best Validation Loss",
        "Baseline": min(baseline_history.history["val_loss"]),
        "Final Improved": min(final_history.history["val_loss"])
    },
    {
        "Metric": "Epochs Trained",
        "Baseline": len(baseline_history.history["loss"]),
        "Final Improved": len(final_history.history["loss"])
    },
    {
        "Metric": "Training Time (s)",
        "Baseline": baseline_time,
        "Final Improved": final_time
    }
])

before_after

## 31. Make predictions

A trained classifier should be used on data that was not used for parameter fitting. We can inspect a few CIFAR-10 test images and compare predicted and actual labels.

In [ ]:
num_examples = 12
sample_images = x_test[:num_examples]
sample_labels = y_test[:num_examples]
probabilities = final_model.predict(sample_images, verbose=0)
predicted_labels = np.argmax(probabilities, axis=1)

fig, axes = plt.subplots(3, 4, figsize=(10, 8))
for i, ax in enumerate(axes.flat):
    ax.imshow(sample_images[i])
    actual = class_names[sample_labels[i]]
    predicted = class_names[predicted_labels[i]]
    ax.set_title(f"Actual: {actual}\nPredicted: {predicted}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 32. Final conceptual map

### Optimization: How do parameters learn?

- Gradient Descent
- SGD / mini-batch SGD
- Momentum
- NAG
- AdaGrad
- AdaDelta
- RMSProp
- Adam

### Training stability: How can learning become easier to control?

- Batch Normalization
- Appropriate learning rate
- Gradient diagnostics

### Generalization: How can we reduce overfitting?

- Dropout
- Early Stopping

### Hyperparameter selection: How do we choose settings?

- Manual tuning
- Grid Search
- Random Search
- Bayesian Optimization
- KerasTuner

The key distinction is:

> **Optimization controls parameter updates. Regularization controls model complexity or training behavior to improve generalization. Normalization stabilizes activations. Hyperparameter tuning searches for useful training and architecture settings.**

## 33. What should you remember from this class?

1. A neural network learns by minimizing a loss function through parameter updates.
2. The learning rate controls the scale of those updates.
3. Momentum adds information from previous updates.
4. NAG evaluates a look-ahead direction.
5. AdaGrad, AdaDelta, RMSProp, and Adam adapt updates using gradient history in different ways.
6. Adam is a strong baseline, but its observed performance should be established experimentally.
7. Batch normalization and optimizers solve different problems.
8. Dropout and early stopping are regularization strategies rather than optimizers.
9. Validation data is used to guide model and hyperparameter choices; the test set should remain a final evaluation set.
10. Hyperparameter tuning is a search problem, and automated methods can reduce manual trial and error.

## 34. Common interview questions

### 1. What is an optimizer?
An optimizer is an algorithm that uses gradient information to update trainable model parameters with the goal of reducing the loss.

### 2. What is the difference between SGD and Momentum?
SGD uses the current gradient directly, while momentum also carries information from previous updates.

### 3. What is NAG?
Nesterov Accelerated Gradient is a momentum-based method that evaluates the gradient at a look-ahead position.

### 4. What problem does AdaGrad address?
It gives parameters adaptive learning rates based on accumulated historical squared gradients.

### 5. Why can AdaGrad's learning rate become very small?
Because its accumulated squared-gradient quantity can keep increasing.

### 6. How does RMSProp address this behavior?
It uses an exponentially weighted moving average of recent squared gradients instead of an ever-growing sum.

### 7. What is the basic idea behind Adam?
Adam combines first-moment information similar to momentum with second-moment information similar to RMSProp.

### 8. Is Adam always the best optimizer?
No. Its performance depends on the dataset, architecture, hyperparameters, and training setup. It should be evaluated empirically.

### 9. What is batch normalization?
It normalizes layer activations using mini-batch statistics and learns scale and shift parameters.

### 10. What is dropout?
Dropout randomly removes a fraction of activations during training as a regularization technique.

### 11. What is early stopping?
It stops training when a monitored validation metric stops improving according to a configured patience value.

### 12. What is the difference between a parameter and a hyperparameter?
Parameters are learned during training; hyperparameters are selected outside the normal gradient-update process.

### 13. Why should the test set not be used repeatedly for hyperparameter tuning?
Repeatedly using the test set can make model-selection decisions indirectly depend on it, reducing its value as an unbiased final evaluation.

### 14. What is grid search?
It evaluates combinations from a predefined grid of hyperparameter values.

### 15. Why can random search be more efficient than grid search?
It can explore more distinct values in important dimensions without spending the same number of trials evaluating every combination in a grid.

### 16. What is Bayesian optimization?
It uses a probabilistic model of the search objective to select promising configurations based on previous trials.

## 35. Instructor checklist

Before class:

- Confirm TensorFlow is installed.
- Confirm CIFAR-10 can be downloaded or is already cached.
- Run the configuration cell and choose `quick` mode.
- Decide whether optimizer comparison should use the quick or full configuration.
- Keep `RUN_TUNER = False` for the main classroom flow unless GPU/time is available.

During class:

1. Run the baseline first.
2. Ask students to predict what changing the optimizer should do.
3. Run the controlled optimizer comparison.
4. Interpret curves rather than simply reading the highest number.
5. Introduce batch normalization, dropout, and early stopping as separate ideas.
6. Discuss vanishing/exploding gradients.
7. Finish with hyperparameter tuning and the final improved model.

After class:

- Switch to `RUN_MODE = "full"`.
- Increase epochs.
- Run KerasTuner random search.
- Compare the selected configuration with the baseline.

## End of notebook

The main lesson is not to memorize optimizer names. The goal is to understand the training problem, isolate one change at a time, measure its effect on validation data, and then combine appropriate techniques into a better model.